# Goal for this notebook 
Create a circulized plot of my genome :
1. 349 bp repeat regions --> bed file
3. Tandem repeat (trf) --> bed file 
4. Interspersed repeat (repeatMasker) --> Bed file
5. Coverage file 
6. Segmental duplication (biser)

In [ ]:
import pandas as pd
from pycirclize import Circos
from pycirclize.parser import Genbank
from pycirclize.utils import load_prokaryote_example_file,  ColorCycler
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
from pycirclize.parser import Gff
ColorCycler.set_cmap("Set3")
from Bio.SeqFeature import SeqFeature, FeatureLocation
from pycirclize.utils import load_eukaryote_example_dataset
from pycirclize import Circos
from matplotlib.colors import to_rgba
from matplotlib.colors import to_hex, to_rgba
import seaborn as sns
import pyranges as pr

In [ ]:


# Load Genbank file
gbk_file = load_prokaryote_example_file("escherichia_coli.gbk.gz")
gbk = Genbank(gbk_file)

# Initialize circos instance
seqid2size = gbk.get_seqid2size()
space = 0 if len(seqid2size) == 1 else 2
circos = Circos(sectors=seqid2size, space=space)
circos.text("Escherichia coli\n(NC_000913)", size=12, r=20)

seqid2features = gbk.get_seqid2features(feature_type=None)
seqid2seq = gbk.get_seqid2seq()
for sector in circos.sectors:
    # Plot outer track with xticks
    major_ticks_interval = 500000
    minor_ticks_interval = 100000
    outer_track = sector.add_track((98, 100))
    outer_track.axis(fc="lightgrey")
    if sector.size >= major_ticks_interval:
        outer_track.xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{v/ 10 ** 6:.1f} Mb"
        )
        outer_track.xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

    f_cds_track = sector.add_track((90, 97), r_pad_ratio=0.1)
    r_cds_track = sector.add_track((83, 90), r_pad_ratio=0.1)
    rrna_track = sector.add_track((76, 83), r_pad_ratio=0.1)
    trna_track = sector.add_track((69, 76), r_pad_ratio=0.1)

    # Plot Forward CDS, Reverse CDS, rRNA, tRNA
    features = seqid2features[sector.name]
    for feature in features:
        if feature.type == "CDS" and feature.location.strand == 1:
            f_cds_track.genomic_features(feature, fc="red")
        elif feature.type == "CDS" and feature.location.strand == -1:
            r_cds_track.genomic_features(feature, fc="blue")
        elif feature.type == "rRNA":
            rrna_track.genomic_features(feature, fc="green")
        elif feature.type == "tRNA":
            trna_track.genomic_features(feature, color="magenta", lw=0.1)

    # Plot GC content
    gc_content_track = sector.add_track((50, 65))
    seq = seqid2seq[sector.name]
    label_pos_list, gc_contents = gbk.calc_gc_content(seq=seq)
    gc_contents = gc_contents - gbk.calc_genome_gc_content(seq=gbk.full_genome_seq)
    positive_gc_contents = np.where(gc_contents > 0, gc_contents, 0)
    negative_gc_contents = np.where(gc_contents < 0, gc_contents, 0)
    abs_max_gc_content = np.max(np.abs(gc_contents))
    vmin, vmax = -abs_max_gc_content, abs_max_gc_content
    gc_content_track.fill_between(
        label_pos_list, positive_gc_contents, 0, vmin=vmin, vmax=vmax, color="black"
    )
    gc_content_track.fill_between(
        label_pos_list, negative_gc_contents, 0, vmin=vmin, vmax=vmax, color="grey"
    )

    # Plot GC skew
    gc_skew_track = sector.add_track((35, 50))

    label_pos_list, gc_skews = gbk.calc_gc_skew(seq=seq)
    positive_gc_skews = np.where(gc_skews > 0, gc_skews, 0)
    negative_gc_skews = np.where(gc_skews < 0, gc_skews, 0)
    abs_max_gc_skew = np.max(np.abs(gc_skews))
    vmin, vmax = -abs_max_gc_skew, abs_max_gc_skew
    gc_skew_track.fill_between(
        label_pos_list, positive_gc_skews, 0, vmin=vmin, vmax=vmax, color="olive"
    )
    gc_skew_track.fill_between(
        label_pos_list, negative_gc_skews, 0, vmin=vmin, vmax=vmax, color="purple"
    )

fig = circos.plotfig()

# Add legend
handles = [
    Patch(color="red", label="Forward CDS"),
    Patch(color="blue", label="Reverse CDS"),
    Patch(color="green", label="rRNA"),
    Patch(color="magenta", label="tRNA"),
    Line2D([], [], color="black", label="Positive GC Content", marker="^", ms=6, ls="None"),
    Line2D([], [], color="grey", label="Negative GC Content", marker="v", ms=6, ls="None"),
    Line2D([], [], color="olive", label="Positive GC Skew", marker="^", ms=6, ls="None"),
    Line2D([], [], color="purple", label="Negative GC Skew", marker="v", ms=6, ls="None"),
]
_ = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.475), loc="center", fontsize=8)

In [ ]:


# Load Genbank file
gbk_file = load_prokaryote_example_file("escherichia_coli.gbk.gz") ## Page for ecoli
gbk = Genbank(gbk_file) # pycirclize.parser.genbank.Genbank

# Initialize circos instance
seqid2size = gbk.get_seqid2size() # Total size of the genome
space = 0 if len(seqid2size) == 1 else 2 # # of genome = 1 
circos = Circos(sectors=seqid2size, space=space)
circos.text("Escherichia coli\n(NC_000913)", size=12, r=20)

seqid2features = gbk.get_seqid2features(feature_type=None)
seqid2seq = gbk.get_seqid2seq()
for sector in circos.sectors:
    # Plot outer track with xticks
    major_ticks_interval = 500000
    minor_ticks_interval = 100000
    outer_track = sector.add_track((98, 100))
    outer_track.axis(fc="lightgrey")
    if sector.size >= major_ticks_interval:
        outer_track.xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{v/ 10 ** 6:.1f} Mb"
        )
        outer_track.xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

    f_cds_track = sector.add_track((90, 97), r_pad_ratio=0.1)
    r_cds_track = sector.add_track((83, 90), r_pad_ratio=0.1)
    rrna_track = sector.add_track((76, 83), r_pad_ratio=0.1)
    trna_track = sector.add_track((69, 76), r_pad_ratio=0.1)

    # Plot Forward CDS, Reverse CDS, rRNA, tRNA
    features = seqid2features[sector.name]
    for feature in features:
        if feature.type == "CDS" and feature.location.strand == 1:
            f_cds_track.genomic_features(feature, fc="red")
        elif feature.type == "CDS" and feature.location.strand == -1:
            r_cds_track.genomic_features(feature, fc="blue")
        elif feature.type == "rRNA":
            rrna_track.genomic_features(feature, fc="green")
        elif feature.type == "tRNA":
            trna_track.genomic_features(feature, color="magenta", lw=0.1)

#     # Plot GC content
    gc_content_track = sector.add_track((50, 65))
    seq = seqid2seq[sector.name]
    label_pos_list, gc_contents = gbk.calc_gc_content(seq=seq)
    gc_contents = gc_contents - gbk.calc_genome_gc_content(seq=gbk.full_genome_seq)
    positive_gc_contents = np.where(gc_contents > 0, gc_contents, 0)
    negative_gc_contents = np.where(gc_contents < 0, gc_contents, 0)
    abs_max_gc_content = np.max(np.abs(gc_contents))
    vmin, vmax = -abs_max_gc_content, abs_max_gc_content
    gc_content_track.fill_between(
        label_pos_list, positive_gc_contents, 0, vmin=vmin, vmax=vmax, color="black"
    )
    gc_content_track.fill_between(
        label_pos_list, negative_gc_contents, 0, vmin=vmin, vmax=vmax, color="grey"
    )

    # Plot GC skew
    gc_skew_track = sector.add_track((35, 50))

    label_pos_list, gc_skews = gbk.calc_gc_skew(seq=seq)
    positive_gc_skews = np.where(gc_skews > 0, gc_skews, 0)
    negative_gc_skews = np.where(gc_skews < 0, gc_skews, 0)
    abs_max_gc_skew = np.max(np.abs(gc_skews))
    vmin, vmax = -abs_max_gc_skew, abs_max_gc_skew
    gc_skew_track.fill_between(
        label_pos_list, positive_gc_skews, 0, vmin=vmin, vmax=vmax, color="olive"
    )
    gc_skew_track.fill_between(
        label_pos_list, negative_gc_skews, 0, vmin=vmin, vmax=vmax, color="purple"
    )

fig = circos.plotfig()

# Add legend
handles = [
    Patch(color="red", label="Forward CDS"),
    Patch(color="blue", label="Reverse CDS"),
    Patch(color="green", label="rRNA"),
    Patch(color="magenta", label="tRNA"),
    Line2D([], [], color="black", label="Positive GC Content", marker="^", ms=6, ls="None"),
    Line2D([], [], color="grey", label="Negative GC Content", marker="v", ms=6, ls="None"),
    Line2D([], [], color="olive", label="Positive GC Skew", marker="^", ms=6, ls="None"),
    Line2D([], [], color="purple", label="Negative GC Skew", marker="v", ms=6, ls="None"),
]
# _ = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.475), loc="center", fontsize=8)

Check out the genome features: awk '{print $3}' hifiasm-041425-scaffolded-chrAssigned.gff | sort | uniq -c

In [ ]:
cytoband_file

In [ ]:
from pycirclize.utils import load_eukaryote_example_dataset

# Load hg38 dataset (https://github.com/moshi4/pycirclize-data/tree/main/eukaryote/hg38)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")

# Initialize Circos from BED chromosomes
circos = Circos.initialize_from_bed(chr_bed_file, space=3)
# circos.text("Homo sapiens (hg38)", size=15)

# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)

# Plot chromosome name
for sector in circos.sectors:
    sector.text(sector.name, size=10)

fig = circos.plotfig()

In [ ]:
from pycirclize import Circos
from pycirclize.utils import ColorCycler, load_eukaryote_example_dataset

# Load hg38 dataset (https://github.com/moshi4/pycirclize-data/tree/main/eukaryote/hg38)
chr_bed_file, cytoband_file, chr_links = load_eukaryote_example_dataset("hg38")

# Initialize Circos from BED chromosomes
circos = Circos.initialize_from_bed(chr_bed_file, space=3)
circos.text("Homo sapiens\n(hg38)", deg=315, r=150, size=12)

# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)

# Create chromosome color dict
ColorCycler.set_cmap("hsv")
chr_names = [s.name for s in circos.sectors]
colors = ColorCycler.get_color_list(len(chr_names))
chr_name2color = {name: color for name, color in zip(chr_names, colors)}

# Plot chromosome name & xticks
for sector in circos.sectors:
    sector.text(sector.name, r=120, size=10, color=chr_name2color[sector.name])
    sector.get_track("cytoband").xticks_by_interval(
        40000000,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v / 1000000:.0f} Mb",
    )

# Plot chromosome link
for link in chr_links:
    region1 = (link.query_chr, link.query_start, link.query_end)
    region2 = (link.ref_chr, link.ref_start, link.ref_end)
    color = chr_name2color[link.query_chr]
    if link.query_chr in ("chr1", "chr8", "chr16") and link.query_chr != link.ref_chr:
        circos.link(region1, region2, color=color)

fig = circos.plotfig()

In [ ]:
karyotype

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

# 4. Read GC content data
gc_content = pd.read_csv(
    "assembly_final.sorted.headerRenamed.chrAssigned.mito.gc_content.bed",
    sep=" ",
    header=None,
    names=["chr", "start", "end", "gc_value"]
)


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################
df_gc = pd.read_csv(
    "assembly_final.sorted.headerRenamed.chrAssigned.mito.gc_content.bed",
    sep=r"\s+",
    header=None,
    names=["chrom", "start", "end", "gc_frac"]
)
# 4. Plot GC track
for sector in circos.sectors:
    # subset to this chromosome
    sub = df_gc[df_gc["chrom"] == sector.name]
    # midpoint of each window
    x = (sub["start"] + sub["end"]) / 2
    y = sub["gc_frac"].values

    # if you want to center on mean:
    y_centered = y - y.mean()

    # create a radial track for GC
    gc_track = sector.add_track((90, 85))
    gc_track.axis(fc="none", ec="grey", lw=0.5)

    # fill above/below zero
    pos = np.where(y_centered > 0, y_centered, 0)
    neg = np.where(y_centered < 0, y_centered, 0)
    max_dev = np.max(np.abs(y_centered))

    gc_track.fill_between(x.values, pos, 0,
                          vmin=-max_dev, vmax=max_dev,
                          color="red")
    gc_track.fill_between(x.values, neg, 0,
                          vmin=-max_dev, vmax=max_dev,
                          color="blue")

    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )
    # outer.xticks_by_interval(50_000_000, tick_length=3,
    #                          outer=True, show_label=True,
    #                          label_formatter=lambda v: f"{v/1e6:.0f} Mb")
    # outer.xticks_by_interval(10_000_000, tick_length=1,
    #                          outer=True, show_label=False)

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
fig.tight_layout()
fig.show()

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

# 1. Read your long-read coverage TSV
df_cov = pd.read_csv(
    "../feature-overview/coverage_long_read_hifiasm_041425_scaffolded_juiceBox_sorted_hardMasked_chrAssigned_15kb_windows.tsv",
    sep=r"\s+",
    header=None,
    names=["chrom", "start", "end", "coverage"]
)
###############################################################

# 2. Define aggregation bin size (e.g., 1 Mb)
bin_size = 1_000_000

# 3. Assign each 15 kb window to a 1 Mb bin
df_cov["bin"] = (df_cov["start"] // bin_size)

# 4. Group by chrom & bin, compute min(start), max(end), mean(coverage)
#    Use as_index=False so 'chrom' and 'bin' become normal columns
df_agg = (
    df_cov
    .groupby(["chrom", "bin"], as_index=False)
    .agg(
        start=("start", "min"),
        end  =("end",   "max"),
        coverage=("coverage", "mean")
    )
)
# 3. Cap coverage at 400
df_agg["coverage_capped"] = np.minimum(
    df_agg["coverage"],
    200                                         # cap value :contentReference[oaicite:6]{index=6}
)

# Now df_agg columns are: ['chrom','bin','start','end','coverage']

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

# 1. Read your long-read coverage TSV
df_cov = pd.read_csv(
    "../feature-overview/coverage_long_read_hifiasm_041425_scaffolded_juiceBox_sorted_chrAssigned_15kb_windows.tsv",
    sep=r"\s+",
    header=None,
    names=["chrom", "start", "end", "coverage"]
)
###############################################################

# 2. Define aggregation bin size (e.g., 1 Mb)
bin_size = 1_000_000

# 3. Assign each 15 kb window to a 1 Mb bin
df_cov["bin"] = (df_cov["start"] // bin_size)

# 4. Group by chrom & bin, compute min(start), max(end), mean(coverage)
#    Use as_index=False so 'chrom' and 'bin' become normal columns
df_agg = (
    df_cov
    .groupby(["chrom", "bin"], as_index=False)
    .agg(
        start=("start", "min"),
        end  =("end",   "max"),
        coverage=("coverage", "mean")
    )
)

cov_threshold=df_agg['coverage'].mean()*2
# 3. Cap coverage at 400
df_agg["coverage_capped"] = np.minimum(
    df_agg["coverage"],
    cov_threshold                                       # cap value :contentReference[oaicite:6]{index=6}
)

# Now df_agg columns are: ['chrom','bin','start','end','coverage']
df_agg.head()
# 6. Add coverage track per sector
for sector in circos.sectors:
    # Subset aggregated coverage for this chromosome
    sub = df_agg[df_agg["chrom"] == sector.name]
    # Midpoints of each aggregate bin
    x = (sub["start"] + sub["end"]) / 2
    y = sub["coverage_capped"].values

    # Create a radial track for coverage
    cov_track = sector.add_track((75, 80))
    cov_track.axis(fc="none", ec="grey", lw=0.5)

    # Fill area under the coverage curve
    cov_track.fill_between(
        x.values, y, 0,
        vmin=0,
        vmax=cov_threshold,           # use the same cap for color scaling if desired
        color="blue"
    )

    # (Optional) Re-draw ticks on outermost track
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        interval=50_000_000,
        tick_length=3,
        outer=True,
        show_label=True,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb"
    )

    # Chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=mid,
        r=115,
        adjust_rotation=True,
        size=10
    )

# 7. Render the figure
fig2 = circos.plotfig()
fig2.tight_layout()
fig2.show()

In [ ]:
final_density=pd.read_csv("segdup_density_rolling_5000000_100000.tsv",sep="\t")
final_density

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    sub_density = final_density[final_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="grey", alpha=0.5, ec="none"
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    sub_density = final_density[final_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="gold", alpha=0.5, ec="none"
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:

# Load repeat data (adjust path/format as needed)
df_repeats_orig = pd.read_csv(
    "assembly_final.sorted.headerRenamed.fasta.out.gff",
    sep="\t",
    header=None,
    names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"],
    comment="#"
)
# df_repeats=df_repeats[df_repeats['chrom'].str.contains('|'.join(["chr1","chr2","chr3","chr4"]))]
print(df_repeats_orig.shape)

In [ ]:
seq_to_chr = {"seq1": "chrX", "seq23": "chrY"}
seq_to_chr.update({f"seq{i}": f"chr{i-1}" for i in list(range(2, 23))})
seq_to_chr.update({f"seq{i}": f"chr{i-2}" for i in list(range(24, 31))})

seq_to_chr

In [ ]:
# Replace values using the dictionary
df_repeats=df_repeats_orig.copy()
df_repeats["chrom"] = df_repeats["chrom"].map(seq_to_chr)
df_repeats = df_repeats.dropna(subset=["chrom"])

In [ ]:
df_repeats.to_csv( "assembly_final.sorted.headerRenamed.fasta.out.chr.gff",header=False,index=False,sep="\t")

In [ ]:
## Subset to 0.01% 
subnum=int(0.0001*len(df_repeats))
df_repeats=df_repeats.sample(n=subnum, random_state=28)

In [ ]:
df_repeats.to_csv( "assembly_final.sorted.headerRenamed.fasta.out.001perc.gff",header=False,index=False,sep="\t")

In [ ]:
## subset for testing
df_repeats=df_repeats.loc[df_repeats['chrom'].isin(["chr1","chr2","chr3","chr4"]),:]
df_repeats

In [ ]:
df_repeats.to_csv( "assembly_final.sorted.headerRenamed.fasta.out.chrSubset.gff",header=False,index=False,sep="\t")

In [ ]:
df_repeats=df_repeats.sample(n=500, random_state=28)

In [ ]:
df_repeats.to_csv( "assembly_final.sorted.headerRenamed.fasta.out.chrSubset500.gff",header=False,index=False,sep="\t")

In [ ]:
gff_repeat = Gff("../feature-overview/assembly_final.sorted.headerRenamed.fasta.out.chrSubset.gff")
gff_repeat

In [ ]:
seqid2features = gff_repeat.get_seqid2features(feature_type=None)
seqid2features

In [ ]:
len(seqid2features['chr2'])

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    repeat_track = sector.add_track((60, 90), r_pad_ratio=0.1)
    feature = seqid2features[sector.name]
    repeat_track.genomic_features(feature, fc="red",ec="red", lw=1, alpha=0.1)



fig = circos.plotfig()
fig.tight_layout()

In [ ]:
df_repeats= pd.read_csv( "../feature-overview/assembly_final.sorted.headerRenamed.fasta.out.chr.gff",
                        names=["chrom","source","type","start","end","score","strand","phase","attributes"],
                        sep="\t")

In [ ]:
# Check if any repeats overlap with each other
print("Checking for overlapping repeats...")

# Sort by chromosome and start position
df_repeats_sorted = df_repeats.sort_values(['chrom', 'start'])

overlapping_count = 0
for chrom in df_repeats_sorted['chrom'].unique():
    chrom_repeats = df_repeats_sorted[df_repeats_sorted['chrom'] == chrom]
    
    for i in range(1, len(chrom_repeats)):
        prev_end = chrom_repeats.iloc[i-1]['end']
        curr_start = chrom_repeats.iloc[i]['start']
        
        if curr_start <= prev_end:
            overlapping_count += 1
            if overlapping_count <= 2:  # Show first 5 examples
                print(f"Overlap found on {chrom}:")
                print(f"  Repeat {i-1}: {chrom_repeats.iloc[i-1]['start']}-{chrom_repeats.iloc[i-1]['end']}")
                print(f"  Repeat {i}: {chrom_repeats.iloc[i]['start']}-{chrom_repeats.iloc[i]['end']}")
                print(f"  Overlap: {prev_end - curr_start + 1} bp")
                print()
            else: 
                break

print(f"Total overlapping repeat pairs: {overlapping_count}")

In [ ]:
# Convert to pyranges
repeats_gr = pr.PyRanges(df_repeats.rename(columns={
    'chrom': 'Chromosome', 
    'start': 'Start', 
    'end': 'End'
}))

# Merge overlapping intervals
merged_gr = repeats_gr.merge()

# Convert back to DataFrame
df_merged = merged_gr.df.rename(columns={
    'Chromosome': 'chrom',
    'Start': 'start',
    'End': 'end'
})
df_merged['length'] = df_merged['end'] - df_merged['start'] + 1

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

## Creating rolling window 
window_size = 1_000_000  # 1 Mb window
step_size = 500_000      # 500 kb step (50% overlap)

# Create rolling windows
windows = []
for chrom, length in chr_sizes.items():
    # For rolling windows, we stop when window would exceed chromosome length
    starts = np.arange(1, length - window_size + 2, step_size)  # +2 to be inclusive
    
    for start in starts:
        end = min(start + window_size - 1, length)  # Inclusive end
        windows.append([chrom, start, end])

windows_df = pd.DataFrame(windows, columns=['Chromosome', 'Start', 'End'])
print(f"Created {len(windows_df)} rolling windows")
print("Sample windows:")
print(windows_df.head(10))

windows_gr = pr.PyRanges(windows_df)

In [ ]:
# Calculate repeat density
repeats_gr = pr.PyRanges(df_merged.rename(columns={
    'chrom': 'Chromosome', 
    'start': 'Start', 
    'end': 'End'
}))

result = windows_gr.join(repeats_gr)
result = result.apply(lambda df: df.assign(
    Overlap=np.minimum(df.End, df.End_b) - np.maximum(df.Start, df.Start_b) + 1
))

density = result.df.groupby(['Chromosome', 'Start', 'End'])['Overlap'].sum().reset_index()
final_density = windows_df.merge(density, on=['Chromosome', 'Start', 'End'], how='left')
final_density['Overlap'] = final_density['Overlap'].fillna(0)
final_density['density'] = final_density['Overlap'] / window_size

# Add midpoint for plotting
final_density['midpoint'] = (final_density['Start'] + final_density['End']) // 2

In [ ]:
repeats_gr

In [ ]:
result

In [ ]:
final_density

In [ ]:
final_density.to_csv("assembly_final.sorted.headerRenamed.repeatDensity.1mb_500kStep_rollingWindow.tsv",sep="\t",index=False)

In [ ]:
final_density=pd.read_csv("assembly_final.sorted.headerRenamed.repeatDensity.1mb_500kStep_rollingWindow.tsv",sep="\t")

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    sub_density = final_density[final_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["density"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="grey", alpha=0.5, ec="none"
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
max(final_density["density"])

In [ ]:
%%time
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    sub_density = final_density[final_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = sub_density["midpoint"].values
    y = sub_density["density"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="grey", alpha=0.5, ec="none"
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
df_tan= pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/trf-tandem-repeat/repeat_df_degu.tsv",
                        sep="\t")

In [ ]:
# Check if any repeats overlap with each other
print("Checking for overlapping repeats...")

# Sort by chromosome and start position
df_tan_sorted = df_tan.sort_values(['sequence', 'start'])

overlapping_count = 0
for chrom in df_tan_sorted['sequence'].unique():
    chrom_repeats = df_tan_sorted[df_tan_sorted['sequence'] == chrom]
    
    for i in range(1, len(chrom_repeats)):
        prev_end = chrom_repeats.iloc[i-1]['end']
        curr_start = chrom_repeats.iloc[i]['start']
        
        if curr_start <= prev_end:
            overlapping_count += 1
            if overlapping_count <= 2:  # Show first 5 examples
                print(f"Overlap found on {chrom}:")
                print(f"  Repeat {i-1}: {chrom_repeats.iloc[i-1]['start']}-{chrom_repeats.iloc[i-1]['end']}")
                print(f"  Repeat {i}: {chrom_repeats.iloc[i]['start']}-{chrom_repeats.iloc[i]['end']}")
                print(f"  Overlap: {prev_end - curr_start + 1} bp")
                print()
            else: 
                break

print(f"Total overlapping repeat pairs: {overlapping_count}")

In [ ]:
# Convert to pyranges
tan_gr = pr.PyRanges(df_tan.rename(columns={
    'sequence': 'Chromosome', 
    'start': 'Start', 
    'end': 'End'
}))

# Merge overlapping intervals
tan_merged_gr = tan_gr.merge()

# Convert back to DataFrame
df_tan_merged = tan_merged_gr.df.rename(columns={
    'Chromosome': 'chrom',
    'Start': 'start',
    'End': 'end'
})
df_tan_merged['length'] = df_tan_merged['end'] - df_tan_merged['start'] + 1

In [ ]:
max(df_tan_merged["length"])

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

## Creating rolling window 
window_size = 1000_000  # 1 Mb window
step_size = 500_000      # 500 kb step (50% overlap)

# Create rolling windows
windows = []
for chrom, length in chr_sizes.items():
    # For rolling windows, we stop when window would exceed chromosome length
    starts = np.arange(1, length - window_size + 2, step_size)  # +2 to be inclusive
    
    for start in starts:
        end = min(start + window_size - 1, length)  # Inclusive end
        windows.append([chrom, start, end])

windows_df = pd.DataFrame(windows, columns=['Chromosome', 'Start', 'End'])
print(f"Created {len(windows_df)} rolling windows")
print("Sample windows:")
print(windows_df.head(10))

windows_gr = pr.PyRanges(windows_df)

In [ ]:
# Calculate repeat density
repeats_tan_gr = pr.PyRanges(df_tan_merged.rename(columns={
    'chrom': 'Chromosome', 
    'start': 'Start', 
    'end': 'End'
}))

result = windows_gr.join(repeats_tan_gr)
result = result.apply(lambda df: df.assign(
    Overlap=np.minimum(df.End, df.End_b) - np.maximum(df.Start, df.Start_b) + 1
))

density_tan = result.df.groupby(['Chromosome', 'Start', 'End'])['Overlap'].sum().reset_index()
final_density_tan = windows_df.merge(density_tan, on=['Chromosome', 'Start', 'End'], how='left')
final_density_tan['Overlap'] = final_density_tan['Overlap'].fillna(0)
final_density_tan['density'] = final_density_tan['Overlap'] / window_size

# Add midpoint for plotting
final_density_tan['midpoint'] = (final_density_tan['Start'] + final_density_tan['End']) // 2

In [ ]:
final_density_tan.to_csv("assembly_final.sorted.headerRenamed.tanRepeatDensity.merged.tsv",sep="\t",index=False)

In [ ]:
plt.hist(final_density_tan["density"])

In [ ]:
plt.hist(final_density_tan.loc[final_density_tan["Chromosome"]=="chr6"]["density"])

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(10).tail(2)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


Y_MAX = 0.1  # Maximum density value that should fill the entire track height
TRACK_BOTTOM = 60
TRACK_TOP = 90
TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
y_max=1
for sector in circos.sectors:


    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue


    density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    # Get your data
    y_data = sub_density["density"].values  # Your values (0 to some max)
    
    # MANUALLY SCALE YOUR DATA - This cancels auto-scaling!
    # y_scaled = TRACK_BOTTOM + (y_data / Y_MAX) * TRACK_HEIGHT
    # # Clip values that would exceed track boundaries
    y_scaled = np.clip(y_data, 0,y_max)
    
    # Plot with manual scaling
    density_track.fill_between(
        x, y_scaled, 0, vmin=0,vmax=y_max,  # Use scaled values and track bottom
        color="grey", alpha=0.5
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(10).tail(2)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


Y_MAX = 0.1  # Maximum density value that should fill the entire track height
TRACK_BOTTOM = 60
TRACK_TOP = 90
TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
y_max = 1

for i, sector in enumerate(circos.sectors):
    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue

    density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
    density_track.axis(fc="none", ec="grey", lw=0.5)  # Adds the track border
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y_data = sub_density["density"].values
    y_scaled = np.clip(y_data, 0, y_max)
    
    density_track.fill_between(
        x, y_scaled, 0, vmin=0, vmax=y_max,
        color="grey", alpha=0.5
    )
    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0, 0.5, 1.0],           # Y-values where ticks should appear
            labels=["0", "0.5", "1.0"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            # label_orientation="vertical", # Orientation of labels
            label_size=10,              # Font size for labels
            # line_width=1,
            # color="black"
        )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(10).tail(2)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


Y_MAX = 0.1  # Maximum density value that should fill the entire track height
TRACK_BOTTOM = 60
TRACK_TOP = 90
TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
y_max = 1

for i, sector in enumerate(circos.sectors):
    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue

    density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
    density_track.axis(fc="none", ec="grey", lw=0.5)  # Adds the track border
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y_data = sub_density["density"].values
    y_scaled = np.clip(y_data, 0, y_max)
    
    # Collapse values in [0.3, 0.9] to a flat line at 0.9
    y_collapsed = np.where((y_scaled >= 0.3) & (y_scaled <= 0.9), 0.9, y_scaled)
    
    density_track.fill_between(
        x, y_collapsed, 0, vmin=0, vmax=y_max,
        color="grey", alpha=0.5
    )
    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0, 0.5, 1.0],           # Y-values where ticks should appear
            labels=["0", "0.5", "1.0"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            # label_orientation="vertical", # Orientation of labels
            label_size=10,              # Font size for labels
            # line_width=1,
            # color="black"
        )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
plt.hist(y_data)

In [ ]:
plt.hist(y_scaled)

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


Y_MAX = 0.1  # Maximum density value that should fill the entire track height
TRACK_BOTTOM = 60
TRACK_TOP = 90
TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
y_max = 1

for i, sector in enumerate(circos.sectors):
    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue

    density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
    density_track.axis(fc="none", ec="grey", lw=0.5)  # Adds the track border
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y_data = sub_density["density"].values
    y_scaled = np.clip(y_data, 0, y_max)
    
    density_track.fill_between(
        x, y_scaled, 0, vmin=0, vmax=y_max,
        color="grey", alpha=0.5
    )
    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0, 0.5, 1.0],           # Y-values where ticks should appear
            labels=["0", "0.5", "1.0"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            # label_orientation="vertical", # Orientation of labels
            label_size=10,              # Font size for labels
            # line_width=1,
            # color="black"
        )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


Y_MAX = 0.1  # Maximum density value that should fill the entire track height
TRACK_BOTTOM = 70
TRACK_TOP = 80
TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
y_max=1
for sector in circos.sectors:


    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue


    density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    # Get your data
    y_data = sub_density["density"].values  # Your values (0 to some max)
    
    # MANUALLY SCALE YOUR DATA - This cancels auto-scaling!
    # y_scaled = TRACK_BOTTOM + (y_data / Y_MAX) * TRACK_HEIGHT
    # # Clip values that would exceed track boundaries
    y_scaled = np.clip(y_data, 0,y_max)
    
    # Plot with manual scaling
    density_track.fill_between(
        x, y_scaled, 0, vmin=0,vmax=y_max,  # Use scaled values and track bottom
        color="grey", alpha=0.5
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
## GPT
# Suppose karyotype is a DataFrame with 'chr' and 'length' columns
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################
# Y_MAX = 0.1  # Maximum density value that should fill the entire track height
# TRACK_BOTTOM = 70
# TRACK_TOP = 80
# TRACK_HEIGHT = TRACK_TOP - TRACK_BOTTOM
# y_max=1
# for sector in circos.sectors:


#     sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
#     if sub_density.empty:  # Skip if no data
#         print(f"No density data for {sector.name}")
#         continue


#     density_track = sector.add_track((TRACK_BOTTOM, TRACK_TOP))
#     density_track.axis(fc="none", ec="grey", lw=0.5)
    
#     x = (sub_density["Start"] + sub_density["End"]) / 2
#     # Get your data
#     y_data = sub_density["density"].values  # Your values (0 to some max)
    
#     # MANUALLY SCALE YOUR DATA - This cancels auto-scaling!
#     # y_scaled = TRACK_BOTTOM + (y_data / Y_MAX) * TRACK_HEIGHT
#     # # Clip values that would exceed track boundaries
#     y_scaled = np.clip(y_data, 0,y_max)
    
#     # Plot with manual scaling
#     density_track.fill_between(
#         x, y_scaled, 0, vmin=0,vmax=y_max,  # Use scaled values and track bottom
#         color="grey", alpha=0.5
#     )
# ------------ #
y_max=1
for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    # sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    # if sub_density.empty:  # Skip if no data
    #     print(f"No density data for {sector.name}")
    #     continue


    # density_track = sector.add_track((60, 70))
    # density_track.axis(fc="none", ec="grey", lw=0.5)
    
    # x = (sub_density["Start"] + sub_density["End"]) / 2
    # y = sub_density["density"].values
    
    # # Clip x-values to sector range to avoid errors
    # x_clipped = np.clip(x, sector.start, sector.end)

    # y_capped = np.clip(y, 0, 0.05)
    # # Normalize y if needed
    # # y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    # density_track.fill_between(
    #     x_clipped, y_capped, 0,  # Use clipped x-values
    #     color="grey", alpha=0.5, ec="none"
    # )

    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue


    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["Start"] + sub_density["End"]) / 2
    # Get your data
    y_data = sub_density["density"].values  # Your values (0 to some max)
    
    # MANUALLY SCALE YOUR DATA - This cancels auto-scaling!
    # y_scaled = TRACK_BOTTOM + (y_data / Y_MAX) * TRACK_HEIGHT
    # # Clip values that would exceed track boundaries
    y_scaled = np.clip(y_data, 0,y_max)
    
    # Plot with manual scaling
    density_track.fill_between(
        x, y_scaled, 0, vmin=0,vmax=y_max,  # Use scaled values and track bottom
        color="grey", alpha=0.5
    )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
final_density_tan

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

## Creating rolling window 
window_size = 1_000_000  # 1 Mb window
step_size = 500_000      # 500 kb step (50% overlap)

# Create rolling windows
windows = []
for chrom, length in chr_sizes.items():
    # For rolling windows, we stop when window would exceed chromosome length
    starts = np.arange(1, length - window_size + 2, step_size)  # +2 to be inclusive
    
    for start in starts:
        end = min(start + window_size - 1, length)  # Inclusive end
        windows.append([chrom, start, end])

windows_df = pd.DataFrame(windows, columns=['Chromosome', 'Start', 'End'])
print(f"Created {len(windows_df)} rolling windows")
print("Sample windows:")
print(windows_df.head(10))

windows_gr = pr.PyRanges(windows_df)

In [ ]:
# Load repeat data (adjust path/format as needed)
df_cent = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/trf-tandem-repeat/349peak_repeat_1millionBpMin.tsv",
    sep="\t", index_col=False)
df_cent = df_cent.iloc[:, 1:]  # Drop first column

print(df_cent.shape)

In [ ]:
gff_feat = Gff("hifiasm_041425_denovoEnhanced_sorted_nc.gff3")

seqid2features_nc = gff_feat.get_seqid2features(feature_type=None)

gff_feat = Gff("hifiasm_041425_denovoEnhanced_sorted_gr.gff3")

seqid2features_gr = gff_feat.get_seqid2features(feature_type=None)

gff_feat = Gff("hifiasm_041425_denovoEnhanced_sorted_ga.gff3")

seqid2features_ga = gff_feat.get_seqid2features(feature_type=None)
seqid2features_ga

In [ ]:
len(seqid2features_nc['chr2'])

In [ ]:
for feature in seqid2features_gr["chr1"]:
    print(feature.type)

In [ ]:
# import pandas as pd
# from pycirclize import Circos
# from pycirclize.parser import Gff
# import matplotlib.pyplot as plt

# # 1. Load chromosome sizes (from .fai or df_cent)
# fai = pd.read_csv(
#     "your_assembly.fasta.fai",
#     sep="\t",
#     header=None,
#     names=["chr", "length", "offset", "linebases", "linewidth"]
# )
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

# 3. Initialize Circos
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

for sector in circos.sectors:
    # Track 1: Chromosome ticks & labels (outer)
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        50_000_000,
        tick_length=3,
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        label_orientation="vertical",
        label_size=8,
        line_kws=dict(ec="grey")
    )
    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        10_000_000,
        tick_length=1,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )
    # Chromosome label
    sector.text(sector.name, size=10, r=115, color="black")

    centromere_track = sector.add_track((60, 65), r_pad_ratio=0.1)
    
    features_this_chr = df_cent[df_cent['chromosome'] == sector.name]
    
    features_to_plot = []
    for _, row in features_this_chr.iterrows():
        feature = SeqFeature(
            FeatureLocation(int(row['start']), int(row['end'])),
            type="repeat",  # or "centromere", or whatever label you prefer
            qualifiers={
                "match_percent": row['match_percent'],
                "repeat_point_relative_perc": row['repeat_point_relative_perc']
            }
        )
        features_to_plot.append(feature)
    
    centromere_track.genomic_features(
        features_to_plot,
        fc="gold",
        ec="goldenrod",
        lw=0.5,
        alpha=0.8
    )

# Render & save
fig = circos.plotfig()
plt.savefig("circos_with_centromeres.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# import pandas as pd
# from pycirclize import Circos
# from pycirclize.parser import Gff
# import matplotlib.pyplot as plt

# # 1. Load chromosome sizes (from .fai or df_cent)
# fai = pd.read_csv(
#     "your_assembly.fasta.fai",
#     sep="\t",
#     header=None,
#     names=["chr", "length", "offset", "linebases", "linewidth"]
# )
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

# 3. Initialize Circos
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

for sector in circos.sectors:
    # Track 1: Chromosome ticks & labels (outer)
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        50_000_000,
        tick_length=3,
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        label_orientation="vertical",
        label_size=8,
        line_kws=dict(ec="grey")
    )
    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        10_000_000,
        tick_length=1,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )
    # Chromosome label
    sector.text(sector.name, size=10, r=115, color="black")

    centromere_track = sector.add_track((60, 65), r_pad_ratio=0.1)
    
    features_this_chr = df_cent[df_cent['chromosome'] == sector.name]
    
    features_to_plot = []
    for _, row in features_this_chr.iterrows():
        feature = SeqFeature(
            FeatureLocation(int(row['start']), int(row['end'])),
            type="repeat",  # or "centromere", or whatever label you prefer
            qualifiers={
                "match_percent": row['match_percent'],
                "repeat_point_relative_perc": row['repeat_point_relative_perc']
            }
        )
        features_to_plot.append(feature)
    
    centromere_track.genomic_features(
        features_to_plot,
        fc="gold",
        ec="goldenrod",
        lw=0.5,
        alpha=0.8
    )

# Render & save
fig = circos.plotfig()
plt.savefig("circos_with_centromeres.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
## Load the chromosomal sizes
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)
# Assuming your junction BED file has columns: chr, start, end, name
hor_bed = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/output/outputs-from-centraAnno/hifiasm-0414/cautils-chrOnly/HORs.bed",
    sep="\t",
    header=0,
    dtype={
        "Sequence_Name": "string",    # Explicit string type
        "HOR_Name": "string",   # 64-bit integer
        "Start_Position": "int64",   # 64-bit integer
        "End_Position": "int64",   # 64-bit integer
        "Num_monomers": "int64",
        "HOR_len": "int64",
        "Span_len":"int64"},     # 64-bit integer
    names=["Sequence_Name", "HOR_Name", 
           "Start_Position", "End_Position",
           "Num_monomers", "HOR_len", "Span_len"],
    na_values=["."],        # Common NA marker in BED files
    keep_default_na=False   # Prevent unwanted NA conversion
)
hor_bed

In [ ]:
# Create the histogram
plt.hist(hor_bed['HOR_len'], bins=1000, edgecolor='black')

# Set the x-axis limits
plt.xlim(0, 1000)

# Set x-axis ticks every 50 units
plt.xticks(np.arange(0, 1001, 50),rotation=90)  # From 0 to 1000 (inclusive), step 50

# Add labels and title
plt.xlabel("HOR length")
plt.ylabel("Frequency")
plt.title("Frequency of HOR length")

# Display the plot
plt.show()

In [ ]:
start=0
end=10000
# Create the histogram
plt.hist(hor_bed['Span_len'], bins=1000, edgecolor='black')

# Set the x-axis limits
plt.xlim(start, end)

# Set x-axis ticks every 50 units
plt.xticks(np.arange(start, end+1, (end-start)/20),rotation=90)  # From 0 to 1000 (inclusive), step 50

# Add labels and title
plt.xlabel("Span length")
plt.ylabel("Frequency")
plt.title("Frequency of Span length")

# Display the plot
plt.show()

In [ ]:
# # Create a sample DataFrame
# data = {'values': np.random.normal(loc=50, scale=10, size=1000)}
# df = pd.DataFrame(data)

# Create the histogram
plt.hist(hor_bed['HOR_len'], bins=1000, edgecolor='black') # 'bins' and 'edgecolor' are optional

# Set the x-axis limits
plt.xlim(0, 1000) # Sets the x-axis from 20 to 80

# Add labels and title (optional)
plt.xlabel("HOR length")
plt.ylabel("Frequency")
plt.title("Frequency of HOR length")

# Display the plot
plt.show()

In [ ]:
# import pandas as pd
# from pycirclize import Circos
# from pycirclize.parser import Gff
# import matplotlib.pyplot as plt

# # 1. Load chromosome sizes (from .fai or df_cent)
# fai = pd.read_csv(
#     "your_assembly.fasta.fai",
#     sep="\t",
#     header=None,
#     names=["chr", "length", "offset", "linebases", "linewidth"]
# )
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

# 3. Initialize Circos
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

for sector in circos.sectors:
    # Track 1: Chromosome ticks & labels (outer)
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        50_000_000,
        tick_length=3,
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        label_orientation="vertical",
        label_size=8,
        line_kws=dict(ec="grey")
    )
    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        10_000_000,
        tick_length=1,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )
    # Chromosome label
    sector.text(sector.name, size=10, r=115, color="black")

    centromere_track = sector.add_track((60, 65), r_pad_ratio=0.1)
    
    features_this_chr = hor_bed[hor_bed['Sequence_Name'] == sector.name]
    
    features_to_plot = []
    for _, row in features_this_chr.iterrows():
        feature = SeqFeature(
            FeatureLocation(int(row['Start_Position']), int(row['End_Position'])),
            # type="repeat",  # or "centromere", or whatever label you prefer
            # qualifiers={
            #     "match_percent": row['match_percent'],
            #     "repeat_point_relative_perc": row['repeat_point_relative_perc']
            # }
        )
        features_to_plot.append(feature)
    
    centromere_track.genomic_features(
        features_to_plot,
        fc="gold",
        ec="goldenrod",
        lw=0.5,
        alpha=0.8
    )

# Render & save
fig = circos.plotfig()
# plt.savefig("circos_with_centromeres.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# import pandas as pd
# from pycirclize import Circos
# from pycirclize.parser import Gff
# import matplotlib.pyplot as plt

# # 1. Load chromosome sizes (from .fai or df_cent)
# fai = pd.read_csv(
#     "your_assembly.fasta.fai",
#     sep="\t",
#     header=None,
#     names=["chr", "length", "offset", "linebases", "linewidth"]
# )
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

# 3. Initialize Circos
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

for sector in circos.sectors:
    # Track 1: Chromosome ticks & labels (outer)
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        50_000_000,
        tick_length=3,
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        label_orientation="vertical",
        label_size=8,
        line_kws=dict(ec="grey")
    )
    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        10_000_000,
        tick_length=1,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )
    # Chromosome label
    sector.text(sector.name, size=10, r=115, color="black")

    centromere_track = sector.add_track((60, 65), r_pad_ratio=0.1)
    
    features_this_chr = hor_bed[hor_bed['Sequence_Name'] == sector.name]
    
    features_to_plot = []
    for _, row in features_this_chr.iterrows():
        feature = SeqFeature(
            FeatureLocation(int(row['Start_Position']), int(row['End_Position'])),
            type="repeat",  # or "centromere", or whatever label you prefer
            # qualifiers={
            #     "match_percent": row['match_percent'],
            #     "repeat_point_relative_perc": row['repeat_point_relative_perc']
            # }
        )
        features_to_plot.append(feature)
    
    centromere_track.genomic_features(
        features_to_plot,
        fc="gold",
        ec="goldenrod",
        lw=0.5,
        alpha=0.8
    )

# Render & save
fig = circos.plotfig()
# plt.savefig("circos_with_centromeres.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
chr_links[:2]

In [ ]:
import pandas as pd

fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)
# Read BEDPE file
bedpe = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_mod.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2"]
)

karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
# Filter for links between the first 4 chromosomes in your karyotype
valid_chrs = list(chr_sizes.keys())  # ['chr1', 'chr2', 'chr3', 'chr4'] or similar
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
].copy()

In [ ]:
chr_colors = {
    'chr1': '#FF6B6B', 'chr2': '#4ECDC4', 'chr3': '#45B7D1', 'chr4': '#FFA07A',
    'chr5': '#92D050', 'chr6': '#D35FB7', 'chr7': '#FFC000', 'chr8': '#00B0F0',
    'chr9': '#A2D96C', 'chr10': '#C00000', 'chr11': '#7030A0', 'chr12': '#FF5733',
    'chr13': '#00AEEF', 'chr14': '#FF99CC', 'chr15': '#8FD8D8', 'chr16': '#F6546A',
    'chr17': '#468499', 'chr18': '#FFD700', 'chr19': '#088DA5', 'chr20': '#F08080',
    'chr21': '#6A5ACD', 'chr22': '#65B891', 'chr23': '#FFA500', 'chr24': '#BA55D3',
    'chr25': '#9370DB', 'chr26': '#3CB371', 'chr27': '#7B68EE', 'chr28': '#40E0D0',
    'chrX': '#FF1493',  # Distinct pink for X
    'chrY': '#14FF82'   # Dark blue for Y
}

# chr_colors = [
#     '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
#     '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
#     '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
#     '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
#     '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173',
#     '#5254a3', '#8ca252', '#bd9e39', '#ad494a', 'black', 
# ]

# 1. Load chromosome sizes
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))


# chromosomes = [f'chr{i}' for i in range(1, 29)] + ['chrX', 'chrY']
# chr_colors = {chrom: colors[i] for i, chrom in enumerate(chromosomes)}

# 2. Initialize Circos plot
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

# 3. Add tracks and ticks
for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    # Major ticks (50 Mb)
    outer.xticks_by_interval(
        interval=50_000_000,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        interval=10_000_000,
        tick_length=1,
        outer=True,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )

    # Chromosome labels
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

# 4. Load and plot BEDPE links
bedpe = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_mod.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2"]
)

# Filter for valid chromosomes
valid_chrs = list(chr_sizes.keys())
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
]


# Plot links
for _, row in bedpe_filtered.iterrows():
    region1 = (row['chr1'], row['start1'], row['end1'])
    region2 = (row['chr2'], row['start2'], row['end2'])
    
    # Use color based on source chromosome with transparency
    color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
    
    # Different linewidths for intra vs inter-chromosomal
    lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
    
    circos.link(
        region1, 
        region2, 
        color=color, 
        direction=1, 
        ec="none", 
        lw=lw
    )

# 5. Finalize plot
fig = circos.plotfig()
fig.tight_layout()
# fig.savefig("circos_plot_with_links.png", dpi=300)

In [ ]:
chr_colors = {
    'chr1': '#FF6B6B', 'chr2': '#4ECDC4', 'chr3': '#45B7D1', 'chr4': '#FFA07A',
    'chr5': '#92D050', 'chr6': '#D35FB7', 'chr7': '#FFC000', 'chr8': '#00B0F0',
    'chr9': '#A2D96C', 'chr10': '#C00000', 'chr11': '#7030A0', 'chr12': '#FF5733',
    'chr13': '#00AEEF', 'chr14': '#FF99CC', 'chr15': '#8FD8D8', 'chr16': '#F6546A',
    'chr17': '#468499', 'chr18': '#FFD700', 'chr19': '#088DA5', 'chr20': '#F08080',
    'chr21': '#6A5ACD', 'chr22': '#65B891', 'chr23': '#FFA500', 'chr24': '#BA55D3',
    'chr25': '#9370DB', 'chr26': '#3CB371', 'chr27': '#7B68EE', 'chr28': '#40E0D0',
    'chrX': '#FF1493',  # Distinct pink for X
    'chrY': '#000080'   # Dark blue for Y
}

# chr_colors = [
#     '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
#     '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
#     '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
#     '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
#     '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173',
#     '#5254a3', '#8ca252', '#bd9e39', '#ad494a', 'black', 
# ]

# 1. Load chromosome sizes
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)
karyotype = fai[fai["chr"].str.contains("chr")].head(2)[["chr", "length"]]
chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))


# chromosomes = [f'chr{i}' for i in range(1, 29)] + ['chrX', 'chrY']
# chr_colors = {chrom: colors[i] for i, chrom in enumerate(chromosomes)}

# 2. Initialize Circos plot
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

# 3. Add tracks and ticks
for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    # Major ticks (50 Mb)
    outer.xticks_by_interval(
        interval=50_000_000,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    # Minor ticks (10 Mb)
    outer.xticks_by_interval(
        interval=10_000_000,
        tick_length=1,
        outer=True,
        show_label=False,
        line_kws=dict(ec="black", lw=0.5)
    )

    # Chromosome labels
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

# 4. Load and plot BEDPE links
bedpe = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_mod.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2"]
)

# Filter for valid chromosomes
valid_chrs = list(chr_sizes.keys())
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
]

# Define the radial positions for your links
r1 = 45  # Inner radius for link starts
r2 = 50  # Outer radius for link ends

# Plot links
for _, row in bedpe_filtered.iterrows():
    region1 = (row['chr1'], row['start1'], row['end1'])
    region2 = (row['chr2'], row['start2'], row['end2'])
    
    # Use color based on source chromosome with transparency
    color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
    
    # Different linewidths for intra vs inter-chromosomal
    lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
    
    circos.link(
        region1, 
        region2, 
        r1=r1,  # Set inner radius
        r2=r2,  # Set outer radius
        color=color, 
        direction=1, 
        ec="none", 
        lw=lw
    )

# 5. Finalize plot
fig = circos.plotfig()
fig.tight_layout()

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

fig = circos.plotfig()
fig.tight_layout()

In [ ]:
new_region = pd.read_csv("df_unaligned.bed", sep="\t",names=["chr","start","end"])

In [ ]:
new_region

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))

window_size = 1_000_000  # 1 Mb windows

density_data = []
for chrom, length in chr_sizes.items():
    bins = np.arange(0, length, window_size)
    for start in bins:
        end = min(start + window_size, length)  # Ensure end ≤ chromosome length
        n_repeats = len(new_region[
            (new_region["chr"] == chrom) & 
            ((new_region["start"] >= end) | 
            (new_region["end"] <= start))
        ])
        density_data.append([chrom, start, end, n_repeats])
df_density_aligned = pd.DataFrame(density_data, columns=["chrom", "start", "end", "count"])

In [ ]:
df_density_aligned

In [ ]:
df_density_aligned.to_csv("df_unaligned_density.tsv",sep="\t",index=False,header=True)

In [ ]:
df_density_aligned[df_density_aligned["count"]!=0]

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################

for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5)

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )

    # sub_density = df_density_aligned[df_density_aligned["chrom"] == sector.name]
    sub_density = df_density_aligned[df_density_aligned["chrom"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((60, 70))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["start"] + sub_density["end"]) / 2
    y = sub_density["count"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,  # Use clipped x-values
        color="grey", alpha=0.5, ec="none"
    )


fig = circos.plotfig()
fig.tight_layout()

In [ ]:
agp=pd.read_csv("out_JBAT_review_with_orig_name.bed", skiprows=2, sep="\t",names=["chrom","start","end","contig","length","strand"])

In [ ]:
agp["chrom_length"]= agp["chrom"].str.split("_").str.get(8).astype(int)

In [ ]:
agp_sorted = agp.sort_values(by=["chrom_length","start"],ascending=[False,True])

In [ ]:
# Step 2: Create a mapping from old chrom names to seq1, seq2, ...
chrom_map = {name: f"seq{i+1}" for i, name in enumerate(agp_sorted["chrom"].unique())}

agp_sorted['chrom'] = agp_sorted['chrom'].map(chrom_map)

In [ ]:
## Step 3 Create mapping for seq1 and seq 23
agp_sorted[agp_sorted["chrom"]=="seq23"] ## Y chrom should be around 86 mbp 

In [ ]:
chrom_map = {
    "seq1": "chrX",
    "seq23": "chrY",
    **{f"seq{i}": f"chr{i-1}" for i in range(2, 23)},  # seq2 → chr2, ..., seq22 → chr22
    **{f"seq{i}": f"chr{i-2}" for i in range(24, 31)}  # seq24 → chr24, ..., seq30 → chr30
}

In [ ]:
agp_sorted['chrom'] = agp_sorted['chrom'].map(chrom_map).fillna(agp_sorted['chrom'])
agp_sorted

In [ ]:
agp_filtered = agp_sorted[agp_sorted['length'].isna()]

In [ ]:
agp_final=agp_filtered.iloc[:,0:3]

In [ ]:
agp_final

In [ ]:
agp_final.to_csv("agp_final_contig2scaffold.bed", sep="\t", index=False)

In [ ]:
# Assuming your junction BED file has columns: chr, start, end, name
junction_bed = pd.read_csv(
    "../feature-overview/agp_final_contig2scaffold.bed",
    sep="\t",
    header=0,
    dtype={
        "chr": "string",    # Explicit string type
        "start": "int64",   # 64-bit integer
        "end": "int64"},     # 64-bit integer
    names=["chr", "start", "end"],
    na_values=["."],        # Common NA marker in BED files
    keep_default_na=False   # Prevent unwanted NA conversion
)

# Filter for only the chromosomes in your karyotype
# junction_bed = junction_bed[junction_bed["chr"].isin(karyotype["chr"])]

In [ ]:
print(junction_bed.dtypes)

In [ ]:
junction_bed

In [ ]:
# 1. Read the .fai file
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]


chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)

###############################################################


for sector in circos.sectors:
    outer = sector.add_track((95, 100), name="axis_track")
    outer.axis(fc="none", ec="black", lw=0.5, capstyle="round")  # Added capstyle

    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5,capstyle="round")  # Rounded minor ticks)
    )

    # Place label at the sector center, just outside the ticks
    center_x = (sector.start + sector.end) / 2
    sector.text(
        text=sector.name,
        x=center_x,
        r=115,
        adjust_rotation=True,
        size=10,
        color="black",
    )


    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")
        


fig = circos.plotfig()
fig.tight_layout()

In [ ]:

fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(4)[["chr", "length"]]

# 3. Set gap degrees (5° after last chromosome)
gap_degrees = 5
gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=10, end=350, space=2, endspace=False)


########################### Segdup density level gene level #########################
sd_gene_density=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_density_rolling_5000000_100000_geneLevel.tsv",sep="\t")

########################### Segdup density level gene level #########################
sd_bp_density=pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/figure/segdup-investigation/segdup_density_rolling_5000000_100000_bpLevel.tsv",sep="\t")

########################### Segmental duplication  ##########################
# Read BEDPE file
bedpe = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_mod.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2"]
)

bedpe = bedpe.sample(frac=0.1, random_state=28)  # random 1000 rows


########################### Add in contig junction information ##########################
# Assuming your junction BED file has columns: chr, start, end, name
junction_bed = pd.read_csv(
    "../feature-overview/agp_final_contig2scaffold.bed",
    sep="\t",
    header=0,
    dtype={
        "chr": "string",    # Explicit string type
        "start": "int64",   # 64-bit integer
        "end": "int64"},     # 64-bit integer
    names=["chr", "start", "end"],
    na_values=["."],        # Common NA marker in BED files
    keep_default_na=False   # Prevent unwanted NA conversion
)

chr_colors = {
    'chr1': '#FF6B6B', 'chr2': '#4ECDC4', 'chr3': '#45B7D1', 'chr4': '#FFA07A',
    'chr5': '#92D050', 'chr6': '#D35FB7', 'chr7': '#FFC000', 'chr8': '#00B0F0',
    'chr9': '#A2D96C', 'chr10': '#C00000', 'chr11': '#7030A0', 'chr12': '#FF5733',
    'chr13': '#00AEEF', 'chr14': '#FF99CC', 'chr15': '#8FD8D8', 'chr16': '#F6546A',
    'chr17': '#468499', 'chr18': '#FFD700', 'chr19': '#088DA5', 'chr20': '#F08080',
    'chr21': '#6A5ACD', 'chr22': '#65B891', 'chr23': '#FFA500', 'chr24': '#BA55D3',
    'chr25': '#9370DB', 'chr26': '#3CB371', 'chr27': '#7B68EE', 'chr28': '#40E0D0',
    'chrX': '#FF1493',  # Distinct pink for X
    'chrY': '#14FF82'   # Dark blue for Y
}

In [ ]:
%%time

y_max=1

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(2)[["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
circos.text("Octodon degus \n assembly", size=12, r=0)


############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):

   



    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")
        
    ### PLOT Segmental duplication #######
    #### Density track for genes in segdup regions
    sub_density = sd_gene_density[sd_gene_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((94, 84))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="blue", alpha=0.5, ec="none"
    )

    ### Density track for bp level in segdup regions
    sub_density = sd_bp_density[sd_bp_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((83, 73))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_segdup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="gold", alpha=0.5, ec="none"
    )

    # Filter for valid chromosomes
    valid_chrs = list(chr_sizes.keys())
    bedpe_filtered = bedpe[
        (bedpe["chr1"].isin(valid_chrs)) & 
        (bedpe["chr2"].isin(valid_chrs))
    ]
    # Define the radial positions for your links
    r1 =72  # Inner radius for link starts
    r2 = 72  # Outer radius for link ends

    # Plot links
    for _, row in bedpe_filtered.iterrows():
        region1 = (row['chr1'], row['start1'], row['end1'])
        region2 = (row['chr2'], row['start2'], row['end2'])
        # Use color based on source chromosome with transparency
        color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
        # Different linewidths for intra vs inter-chromosomal
        lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
        circos.link(
            region1, 
            region2, 
            r1=r1,  # Set inner radius
            r2=r2,  # Set outer radius
            color=color, 
            direction=1, 
            ec="none", 
            lw=lw
        )

    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0.5, 1.0],           # Y-values where ticks should appear
            labels=["0.5", "1"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            label_size=7,              # Font size for labels
        )


# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "1) Chromosomes", "black"),
    (84, "2) Segdup density\n    at gene-level","black"),
    (73, "3) Segdup density\n    at bp-level","black"),
    (55, "4) Segdup links", "black")
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=8,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()

In [ ]:
%%time

y_max=1

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(10)[["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
# circos.text("Octodon degus \n assembly", size=12, r=0)


############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):
    print(sector.name)

    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")
        
    ### PLOT Segmental duplication #######
    #### Density track for genes in segdup regions
    sub_density = sd_gene_density[sd_gene_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((94, 84))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="blue", alpha=0.5, ec="none"
    )

    ### Density track for bp level in segdup regions
    sub_density = sd_bp_density[sd_bp_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((83, 73))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_segdup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="gold", alpha=0.5, ec="none"
    )


    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0.5, 1.0],           # Y-values where ticks should appear
            labels=["0.5", "1"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            label_size=7,              # Font size for labels
        )

# Filter for valid chromosomes
valid_chrs = list(chr_sizes.keys())
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
]
# Define the radial positions for your links
# r1 =72  # Inner radius for link starts
# r2 = 72  # Outer radius for link ends

# Plot links
for _, row in bedpe_filtered.iterrows():
    region1 = (row['chr1'], row['start1'], row['end1'])
    region2 = (row['chr2'], row['start2'], row['end2'])
    # Use color based on source chromosome with transparency
    color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
    # Different linewidths for intra vs inter-chromosomal
    lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
    circos.link(
        region1, 
        region2, 
        # r1=r1,  # Set inner radius
        # r2=r2,  # Set outer radius
        color=color, 
        direction=1, 
        ec="none", 
        lw=lw
    )

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "1) Chromosomes", "black"),
    (84, "2) Segdup density\n    at gene-level","black"),
    (73, "3) Segdup density\n    at bp-level","black"),
    (65, "4) Segdup links", "black")
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=8,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()

In [ ]:
%%time

y_max=1

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")].head(11)[["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
# circos.text("Octodon degus \n assembly", size=12, r=0)


############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):
    print(sector.name)

    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")
        
    ### PLOT Segmental duplication #######
    #### Density track for genes in segdup regions
    sub_density = sd_gene_density[sd_gene_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((94, 84))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="blue", alpha=0.5, ec="none"
    )

    ### Density track for bp level in segdup regions
    sub_density = sd_bp_density[sd_bp_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((83, 73))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_segdup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="gold", alpha=0.5, ec="none"
    )


    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0.5, 1.0],           # Y-values where ticks should appear
            labels=["0.5", "1"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            label_size=7,              # Font size for labels
        )

# Filter for valid chromosomes
valid_chrs = list(chr_sizes.keys())
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
]
# Define the radial positions for your links
# r1 =72  # Inner radius for link starts
# r2 = 72  # Outer radius for link ends

# Plot links
for _, row in bedpe_filtered.iterrows():
    region1 = (row['chr1'], row['start1'], row['end1'])
    region2 = (row['chr2'], row['start2'], row['end2'])
    # Use color based on source chromosome with transparency
    color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
    # Different linewidths for intra vs inter-chromosomal
    lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
    circos.link(
        region1, 
        region2, 
        # r1=r1,  # Set inner radius
        # r2=r2,  # Set outer radius
        color=color, 
        direction=1, 
        ec="none", 
        lw=lw
    )

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "1) Chromosomes", "black"),
    (84, "2) Segdup density\n    at gene-level","black"),
    (73, "3) Segdup density\n    at bp-level","black"),
    (65, "4) Segdup links", "black")
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=8,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()

In [ ]:
%%time

y_max=1

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
# circos.text("Octodon degus \n assembly", size=12, r=0)


############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):
    print(sector.name)

    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")
        
    ### PLOT Segmental duplication #######
    #### Density track for genes in segdup regions
    sub_density = sd_gene_density[sd_gene_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((94, 84))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_dup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="blue", alpha=0.5, ec="none"
    )

    ### Density track for bp level in segdup regions
    sub_density = sd_bp_density[sd_bp_density["Chromosome"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    density_track = sector.add_track((83, 73))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["percent_segdup"].values
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(
        x_clipped, y_norm, 0,vmax=1,  # Use clipped x-values
        color="gold", alpha=0.5, ec="none"
    )


    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

    
    if i == 0:  # First sector only
        density_track.yticks(
            y=[0.5, 1.0],           # Y-values where ticks should appear
            labels=["0.5", "1"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            label_size=7,              # Font size for labels
        )

# Filter for valid chromosomes
valid_chrs = list(chr_sizes.keys())
bedpe_filtered = bedpe[
    (bedpe["chr1"].isin(valid_chrs)) & 
    (bedpe["chr2"].isin(valid_chrs))
]
# Define the radial positions for your links
# r1 =72  # Inner radius for link starts
# r2 = 72  # Outer radius for link ends

# Plot links
for _, row in bedpe_filtered.iterrows():
    region1 = (row['chr1'], row['start1'], row['end1'])
    region2 = (row['chr2'], row['start2'], row['end2'])
    # Use color based on source chromosome with transparency
    color = to_rgba(chr_colors[row['chr1']], alpha=0.3)
    # Different linewidths for intra vs inter-chromosomal
    lw = 0.8 if row['chr1'] == row['chr2'] else 0.5
    circos.link(
        region1, 
        region2, 
        # r1=r1,  # Set inner radius
        # r2=r2,  # Set outer radius
        color=color, 
        direction=1, 
        ec="none", 
        lw=lw
    )

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "1) Chromosomes", "black"),
    (84, "2) Segdup density\n    at gene-level","black"),
    (73, "3) Segdup density\n    at bp-level","black"),
    (65, "4) Segdup links", "black")
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=8,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()

In [ ]:
# Save in PNG (lossless) or PDF/SVG (vector)
fig.savefig(
    "degus_genome_circos_segdup_overview.png",  # For PNG
    dpi=600,               # Ultra-high resolution (300-600 for print)
    bbox_inches="tight",   # Prevents cropping
    transparent=False,     # Set to True if you need transparency
    facecolor="white"      # Background color
)

In [ ]:
# Save in PNG (lossless) or PDF/SVG (vector)
fig.savefig(
    "degus_genome_circos_segdup_overview.pdf",  # For PNG
    dpi=600,               # Ultra-high resolution (300-600 for print)
    bbox_inches="tight",   # Prevents cropping
    transparent=False,     # Set to True if you need transparency
    facecolor="white"      # Background color
)

In [ ]:
# Save in PNG (lossless) or PDF/SVG (vector)
fig.savefig(
    "degus_genome_circos_segdup_overview.svg",  
    dpi=600,               # Ultra-high resolution (300-600 for print)
    bbox_inches="tight",   # Prevents cropping
    transparent=False,     # Set to True if you need transparency
    facecolor="white"      # Background color
)

In [ ]:
%%time

# 2. Filter for chromosomes and keep first 4
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
circos.text("Octodon degus \n assembly", size=12, r=0)

# gff_repeat = Gff("assembly_final.sorted.headerRenamed.fasta.out.001perc.gff")
# seqid2repfeatures = gff_repeat.get_seqid2features(feature_type=None)


############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):
    # Plot forward/reverse CDS, rRNA, tRNA tracks
    rename_cds_track = sector.add_track((85, 95), r_pad_ratio=0.1)
    replace_cds_track = sector.add_track((75, 85), r_pad_ratio=0.1)
    added_cds_track = sector.add_track((65, 75), r_pad_ratio=0.1)
    
    # Check if chromosome exists in each dictionary before processing
    if sector.name in seqid2features_nc:
        for feature in seqid2features_nc[sector.name]:
            if feature.type == "gene":
                rename_cds_track.genomic_features(feature, fc="firebrick", ec="firebrick", lw=0.01)
    
    if sector.name in seqid2features_gr:
        for feature in seqid2features_gr[sector.name]:
            if feature.type == "gene":
                replace_cds_track.genomic_features(feature, fc="limegreen", ec="limegreen", lw=0.01)
    
    if sector.name in seqid2features_ga:
        for feature in seqid2features_ga[sector.name]:
            if feature.type == "gene":
                added_cds_track.genomic_features(feature, fc="royalblue", ec="royalblue", lw=0.01)

    # Alignment density (higher is more unaligned or novel genomic regions)
    sub_density = df_density[df_density["chrom"] == sector.name]
    if sub_density.empty:  # Skip if no data
        print(f"No density data for {sector.name}")
        continue
    
    density_track = sector.add_track((53, 63))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    
    x = (sub_density["start"] + sub_density["end"]) / 2
    y = sub_density["count_norm"].values
    
    # Clip x-values to sector range to avoid errors
    x_clipped = np.clip(x, sector.start, sector.end)
    
    # Normalize y if needed
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    
    density_track.fill_between(
        x_clipped, y_norm, 0,  # Use clipped x-values
        color="grey", alpha=0.5, ec="none"
    )


    ### PLOT COVERAGE
    # Subset aggregated coverage for this chromosome
    sub = df_agg[df_agg["chrom"] == sector.name]
    # Midpoints of each aggregate bin
    x = (sub["start"] + sub["end"]) / 2
    y = sub["coverage_capped"].values
    # Create a radial track for coverage
    cov_track = sector.add_track((46, 51)) #****
    cov_track.axis(fc="none", ec="grey", lw=0.5)
    if i == 0:
        # Add y-axis ticks for density track
        max_density = y.max()
        cov_track.yticks(
            y=[0, max_density/2],
            labels=[0, int(max_density/2)],
            tick_length=1,
            label_size=6,
            line_kws=dict(ec="grey", lw=0.5),
            side='left',
            vmin=0,
            vmax=max_density)
    # Fill area under the coverage curve
    cov_track.fill_between(
        x.values, y, 0,
        vmin=0,
        vmax=cov_threshold,           # use the same cap for color scaling if desired
        color="black"
    )

    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")

    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "Chromosomes", "black"),
    (85, "LOC renamed", "tomato"),
    (75, "LOC replaced", "limegreen"),
    (65, "Denovo added", "dodgerblue"),
    (53, "Unaligned region", "gold"),
    (43.5, "Read cov", "mediumorchid")
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=7,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()

In [ ]:
# Save in PNG (lossless) or PDF/SVG (vector)
fig.savefig(
    "degus_genome_circos_annotation_change_overview.png",  # For PNG
    dpi=600,               # Ultra-high resolution (300-600 for print)
    bbox_inches="tight",   # Prevents cropping
    transparent=False,     # Set to True if you need transparency
    facecolor="white"      # Background color
)

In [ ]:
fai = pd.read_csv(
    "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)

In [ ]:
gff_nc = pd.read_csv("hifiasm_041425_denovoEnhanced_sorted_nc.tsv",sep="\t",names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])
gene_nc=gff_nc[gff_nc["type"]=="gene"]
gff_gr = pd.read_csv("hifiasm_041425_denovoEnhanced_sorted_gr.tsv",sep="\t",names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])
gene_gr=gff_gr[gff_gr["type"]=="gene"]
gff_ga = pd.read_csv("hifiasm_041425_denovoEnhanced_sorted_ga.tsv",sep="\t",names=["chrom", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])
gene_ga=gff_ga[gff_ga["type"]=="gene"]

In [ ]:
gene_ga

In [ ]:
import pandas as pd

# Load your GFF files (assuming they're already loaded as gff1, gff2, gff3)
gffs = [gene_nc, gene_gr, gene_ga]  # List of your three GFF DataFrames

# 1. Get chromosome lengths from fai
chrom_lengths = dict(zip(fai['chr'], fai['length']))

# 2. Find common chromosomes present in all 3 GFFs AND in fai
common_chroms = set(gffs[0]['chrom'].unique()).intersection(
                 set(gffs[1]['chrom'].unique()),
                 set(gffs[2]['chrom'].unique()),
                 set(chrom_lengths.keys()))
common_chroms = sorted(common_chroms)

# 3. Pre-process GFFs for faster window checking
def prepare_gff_windows(gff, window_size=1_000_000):
    """Convert GFF to dictionary of {chrom: set(window_numbers)}"""
    gff_windows = {}
    for chrom in common_chroms:
        chrom_data = gff[gff['chrom'] == chrom]
        windows = set()
        for _, row in chrom_data.iterrows():
            start_window = row['start'] // window_size
            end_window = row['end'] // window_size
            windows.update(range(start_window, end_window + 1))
        gff_windows[chrom] = windows
    return gff_windows

# Create window sets for each GFF
gff_windows = [prepare_gff_windows(gff) for gff in gffs]

# 4. Find overlapping windows
results = []
window_size = 1_000_000

for chrom in common_chroms:
    max_window = chrom_lengths[chrom] // window_size
    # Find intersection of windows present in all 3 GFFs
    common_windows = set(gff_windows[0][chrom])
    for gff in gff_windows[1:]:
        common_windows.intersection_update(gff[chrom])
    
    # Convert window numbers back to genomic coordinates
    for window in sorted(common_windows):
        start = window * window_size
        end = (window + 1) * window_size
        # Handle chromosome ends
        if end > chrom_lengths[chrom]:
            end = chrom_lengths[chrom]
        results.append({
            'chrom': chrom,
            'start': start,
            'end': end,
            'window': window
        })

# 5. Create result DataFrame
result_df = pd.DataFrame(results)
print(f"Found {len(result_df)} 1Mb windows with features in all 3 GFFs")
print(result_df)

In [ ]:
chrom="chr19"
start=40000000
end=41000000
# chrom="chr3"
# start=40000000
# end=41000000
# chrom="chr2"
# start=19000000
# end=20000000

In [ ]:
gene_nc[(gene_nc["chrom"] ==chrom)&(gene_nc["start"]>=start)&(gene_nc["end"]<=end)]

In [ ]:
gene_gr[(gene_gr["chrom"] ==chrom)&(gene_gr["start"]>=start)&(gene_gr["end"]<=end)]

In [ ]:
gene_ga[(gene_ga["chrom"] ==chrom)&(gene_ga["start"]>=start)&(gene_ga["end"]<=end)]